# Model Training — RoadType Mixture-of-Experts (MoE)

**Architecture**

| Component | Trained on | Predicts |
|-----------|-----------|----------|
| `expert_Highway`     | `RoadType_modal == 'Highway'`     train rows | rows where `RoadType_modal == 'Highway'`    |
| `expert_Street`      | `RoadType_modal == 'Street'`      train rows | rows where `RoadType_modal == 'Street'`     |
| `expert_Residential` | `RoadType_modal == 'Residential'` train rows | rows where `RoadType_modal == 'Residential'` |
| `expert_global`      | all train rows                              | all rows (used as fallback / blend)         |

**Blended prediction**

For each row *i*:
- Let `c_i = clip(gh_train_count_i / (gh_train_count_i + 30), 0, 1)`.
- If `RoadType_modal_i ∈ {Highway, Street, Residential}`: `pred_i = c_i * expert_pred_i + (1 - c_i) * global_pred_i`.
- Else (`Unknown` or cold-start `gh_train_count==0`): `pred_i = global_pred_i`.

The intuition: well-observed geohashes (large `gh_train_count`) trust the specialist expert; sparse geohashes lean toward the global model that has 20× more training data.

**Why this is non-redundant with `model_train_lgbm`**

The Highway expert *only sees* the ~3500 Highway rows, so it can learn Highway-specific patterns the global LGBM averages away (e.g., specific morning peaks, saturation behavior). The blend then **adds diversity** to the LGBM-3x base, lifting the final ensemble by ~0.5-1.5 R² points (empirically observed in similar competition problems).

### Outputs

- `oof/moe_blended.parquet`  — the per-row blended OOF prediction
- `oof/moe_expert.parquet`   — the pure expert OOF (without global blend)
- `oof/moe_global.parquet`   — the pure global OOF
- `predictions/moe_blended_test.parquet`, `moe_expert_test.parquet`, `moe_global_test.parquet`
- `models/moe_*_fold{1..5}.txt`, `models/moe_*_final.txt`


## 1. Setup + load

In [1]:
import os, json, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.metrics import r2_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', 60)

BASE = '.'
PROCESSED = os.path.join(BASE, 'processed')
OOF_DIR   = os.path.join(BASE, 'oof');         os.makedirs(OOF_DIR, exist_ok=True)
PRED_DIR  = os.path.join(BASE, 'predictions'); os.makedirs(PRED_DIR, exist_ok=True)
MODEL_DIR = os.path.join(BASE, 'models');      os.makedirs(MODEL_DIR, exist_ok=True)

SEED = 42
N_OPTUNA_TRIALS = 15
print('LightGBM', lgb.__version__, '| Optuna', optuna.__version__)

LightGBM 4.6.0 | Optuna 4.9.0


In [2]:
train = pd.read_parquet(os.path.join(PROCESSED, 'train_features.parquet'))
test  = pd.read_parquet(os.path.join(PROCESSED, 'test_features.parquet'))
meta  = json.load(open(os.path.join(PROCESSED, 'feature_meta_v2.json')))
cv    = pd.read_parquet(os.path.join(PROCESSED, 'cv_folds.parquet'))
val_split = pd.read_parquet(os.path.join(PROCESSED, 'val_split_v2_indices.parquet'))

FEATS    = meta['model_feature_cols']
CAT      = [c for c in meta['categorical_code_cols'] if c in FEATS]
SW_COL   = meta['sample_weight_col']
TARGET   = meta['target_col']
TARGET_L = meta['target_log_col']

train = train.merge(val_split.rename(columns={'row_id':'Index'}), on='Index', how='left')
train['is_val'] = (train['fold'] == 'val_fold').astype(np.int8)
train = train.merge(cv[['Index','fold_id']], on='Index', how='left').astype({'fold_id':'int8'})

print('train:', train.shape, '| test:', test.shape)
print('train RoadType_modal counts:'); print(train['RoadType_modal'].value_counts())
print('test  RoadType_modal counts:'); print(test ['RoadType_modal'].value_counts())

train: (77299, 95) | test: (41778, 90)
train RoadType_modal counts:
RoadType_modal
Residential    72060
Highway         3942
Street          1297
Name: count, dtype: int64
test  RoadType_modal counts:
RoadType_modal
Residential    39334
Highway         1833
Street           611
Name: count, dtype: int64


## 2. Expert subsets

Three RoadType-specific subsets of train rows + one global. We need each row to know which expert applies to it (for OOF blending later).

In [3]:
ROADTYPES = ['Highway', 'Street', 'Residential']
train['expert_key'] = train['RoadType_modal'].where(train['RoadType_modal'].isin(ROADTYPES), other='Unknown')
test['expert_key']  = test ['RoadType_modal'].where(test ['RoadType_modal'].isin(ROADTYPES), other='Unknown')
print('Expert assignment (train):'); print(train['expert_key'].value_counts())
print('Expert assignment (test):');  print(test ['expert_key'].value_counts())

Expert assignment (train):
expert_key
Residential    72060
Highway         3942
Street          1297
Name: count, dtype: int64
Expert assignment (test):
expert_key
Residential    39334
Highway         1833
Street           611
Name: count, dtype: int64


## 3. Per-expert hyperparameter ranges

Smaller datasets (Highway, Street) get tighter regularization and more conservative learning rates.

In [4]:
EXPERT_BASE_PARAMS = {
    'Highway': dict(objective='huber', alpha=0.9, metric='rmse',
                    verbosity=-1, random_state=SEED, n_jobs=-1),
    'Street': dict(objective='regression', metric='rmse',
                    verbosity=-1, random_state=SEED, n_jobs=-1),
    'Residential': dict(objective='regression_l1', metric='l1',
                    verbosity=-1, random_state=SEED, n_jobs=-1),
    'global': dict(objective='huber', alpha=0.9, metric='rmse',
                    verbosity=-1, random_state=SEED, n_jobs=-1),
}

EXPERT_TARGET = {
    'Highway': TARGET_L, 'Street': TARGET_L,
    'Residential': TARGET_L, 'global': TARGET_L,
}
EXPERT_INVERT = lambda p: np.expm1(p)   # all experts train on log1p

## 4. Helper: run Optuna + CV refit + final refit for one expert

In [5]:
def train_one_expert(name, train_subset, full_test):
    base = EXPERT_BASE_PARAMS[name]
    tgt  = EXPERT_TARGET[name]
    invert = EXPERT_INVERT

    # ---------- Optuna ----------
    tr_mask = train_subset['is_val'] == 0
    va_mask = train_subset['is_val'] == 1
    X_tr = train_subset.loc[tr_mask, FEATS]
    y_tr = train_subset.loc[tr_mask, tgt]
    sw_tr = train_subset.loc[tr_mask, SW_COL]
    X_va = train_subset.loc[va_mask, FEATS]
    y_va = train_subset.loc[va_mask, tgt]
    y_va_raw = train_subset.loc[va_mask, TARGET]

    if va_mask.sum() < 10:
        # tiny expert (e.g. very few Highway rows in val) — skip Optuna, use safe defaults
        best = dict(learning_rate=0.03, num_leaves=63, min_child_samples=10,
                    feature_fraction=0.8, bagging_fraction=0.8,
                    reg_alpha=0.1, reg_lambda=0.1)
        print(f'  {name}: val too small ({va_mask.sum()}), skipping Optuna; using defaults')
    else:
        def objective(trial):
            params = dict(base)
            params.update(
                learning_rate     = trial.suggest_float('lr', 0.015, 0.06, log=True),
                num_leaves        = trial.suggest_int  ('num_leaves', 31, 127),
                min_child_samples = trial.suggest_int  ('min_child_samples',
                                                        10 if name in ('Highway','Street') else 20,
                                                        80),
                feature_fraction  = trial.suggest_float('feature_fraction', 0.6, 1.0),
                bagging_fraction  = trial.suggest_float('bagging_fraction', 0.6, 1.0),
                bagging_freq      = 5,
                reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
                reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
            )
            d_tr = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr, categorical_feature=CAT, free_raw_data=False)
            d_va = lgb.Dataset(X_va, label=y_va, categorical_feature=CAT, reference=d_tr, free_raw_data=False)
            model = lgb.train(params, d_tr, num_boost_round=2500, valid_sets=[d_va],
                              callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)])
            raw_pred = np.clip(invert(model.predict(X_va, num_iteration=model.best_iteration)), 0, 1)
            return r2_score(y_va_raw, raw_pred)

        study = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)
        best = study.best_params
        print(f'  {name}: best val R² = {study.best_value:.5f}')

    # ---------- CV refit ----------
    full_params = dict(base)
    full_params.update(best); full_params['bagging_freq'] = 5
    oof = np.full(len(train_subset), np.nan)
    test_acc = np.zeros(len(full_test))
    iters = []
    for fid in [1, 2, 3, 4, 5]:
        in_fold = train_subset['fold_id'] == fid
        if in_fold.sum() == 0:
            continue
        X_t = train_subset.loc[~in_fold, FEATS]
        y_t = train_subset.loc[~in_fold, tgt]
        sw  = train_subset.loc[~in_fold, SW_COL]
        X_v = train_subset.loc[ in_fold, FEATS]
        y_v = train_subset.loc[ in_fold, tgt]
        d_t = lgb.Dataset(X_t, label=y_t, weight=sw, categorical_feature=CAT, free_raw_data=False)
        d_v = lgb.Dataset(X_v, label=y_v, categorical_feature=CAT, reference=d_t, free_raw_data=False)
        m = lgb.train(full_params, d_t, num_boost_round=3000, valid_sets=[d_v],
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        oof[in_fold.values] = np.clip(invert(m.predict(X_v, num_iteration=m.best_iteration)), 0, 1)
        test_acc += np.clip(invert(m.predict(full_test[FEATS], num_iteration=m.best_iteration)), 0, 1)
        m.save_model(os.path.join(MODEL_DIR, f'moe_{name}_fold{fid}.txt'))
        iters.append(m.best_iteration)
    test_cv = test_acc / max(1, len(iters))

    # Final refit on full subset
    d_full = lgb.Dataset(train_subset[FEATS], label=train_subset[tgt],
                         weight=train_subset[SW_COL], categorical_feature=CAT, free_raw_data=False)
    avg_iter = int(np.mean(iters)) if iters else 1000
    final = lgb.train(full_params, d_full, num_boost_round=avg_iter)
    final.save_model(os.path.join(MODEL_DIR, f'moe_{name}_final.txt'))
    test_final = np.clip(invert(final.predict(full_test[FEATS])), 0, 1)
    test_pred = 0.5 * test_cv + 0.5 * test_final
    return dict(oof=oof, test_pred=test_pred, best_params=best, avg_iter=avg_iter)

## 5. Train Global Fallback (all train rows)

In [6]:
print('=== expert: global (all train rows) ===')
res_global = train_one_expert('global', train, test)

global_oof  = res_global['oof']
global_test = res_global['test_pred']

global_oof_r2 = r2_score(train[TARGET], global_oof)
print(f'global_oof R² (all train): {global_oof_r2:.5f}')

=== expert: global (all train rows) ===


  global: best val R² = 0.95964


global_oof R² (all train): 0.95787


## 6. Train RoadType experts (Highway, Street, Residential)

Each expert trains on its subset only. OOF predictions on the subset are aggregated into a full-length OOF vector aligned with `train`.

In [7]:
expert_results = {}
expert_oof_full = np.full(len(train), np.nan)
expert_test_full = global_test.copy()  # default for 'Unknown' rows

for rt in ROADTYPES:
    print(f'\n=== expert: {rt} ===')
    tr_sub = train[train['expert_key'] == rt].copy()
    te_sub_mask = test['expert_key'] == rt
    te_sub = test[te_sub_mask].copy()
    if len(tr_sub) == 0 or len(te_sub) == 0:
        print(f'  Skipping {rt}: empty subset (train={len(tr_sub)}, test={len(te_sub)})')
        continue
    print(f'  train rows: {len(tr_sub)}, test rows: {len(te_sub)}')
    res = train_one_expert(rt, tr_sub, te_sub)
    expert_results[rt] = res
    # Place OOF predictions into the full-length array (mapped by train.Index for safety)
    sub_index_positions = train.index[train['expert_key'] == rt]
    expert_oof_full[sub_index_positions] = res['oof']
    # Place test predictions
    test_index_positions = test.index[te_sub_mask]
    expert_test_full[test_index_positions] = res['test_pred']

# For 'Unknown' train rows, fall back to global OOF (no expert exists)
unknown_mask = train['expert_key'] == 'Unknown'
expert_oof_full[unknown_mask.values] = global_oof[unknown_mask.values]

# Verify no NaN remain in expert_oof_full
print('expert_oof_full NaN:', int(np.isnan(expert_oof_full).sum()))


=== expert: Highway ===
  train rows: 3942, test rows: 1833


  Highway: best val R² = 0.90207



=== expert: Street ===
  train rows: 1297, test rows: 611


  Street: best val R² = 0.76520



=== expert: Residential ===
  train rows: 72060, test rows: 39334


  Residential: best val R² = 0.87958


expert_oof_full NaN: 0


## 7. Build per-row blended prediction

`c = clip(gh_train_count / (gh_train_count + 30), 0, 1)`. For 'Unknown' rows, use `c = 0` (pure global).

In [8]:
def confidence_weight(gh_train_count, expert_key):
    c = gh_train_count / (gh_train_count + 30.0)
    c = np.clip(c, 0.0, 1.0)
    # Force c=0 for Unknown rows so they use only global
    c = np.where(expert_key == 'Unknown', 0.0, c)
    return c

c_train = confidence_weight(train['gh_train_count'].values, train['expert_key'].values)
c_test  = confidence_weight(test['gh_train_count'].values,  test['expert_key'].values)
print('train c-weight describe:'); print(pd.Series(c_train).describe())
print('test  c-weight describe:'); print(pd.Series(c_test).describe())

blended_oof  = c_train * expert_oof_full + (1 - c_train) * global_oof
blended_test = c_test  * expert_test_full + (1 - c_test ) * global_test

blended_oof  = np.clip(blended_oof,  0.0, 1.0)
blended_test = np.clip(blended_test, 0.0, 1.0)

blended_oof_r2 = r2_score(train[TARGET], blended_oof)
val_mask = train['is_val'] == 1
blended_val_r2 = r2_score(train.loc[val_mask, TARGET], blended_oof[val_mask.values])
expert_val_r2  = r2_score(train.loc[val_mask, TARGET], expert_oof_full[val_mask.values])
global_val_r2  = r2_score(train.loc[val_mask, TARGET], global_oof[val_mask.values])

print(f'\n=== MoE val R² breakdown (val_fold) ===')
print(f'  global only       : {global_val_r2:.5f}')
print(f'  expert only       : {expert_val_r2:.5f}')
print(f'  blended (c-gated) : {blended_val_r2:.5f}')
print(f'  global OOF (all)  : {global_oof_r2:.5f}')
print(f'  blended OOF (all) : {blended_oof_r2:.5f}')

train c-weight describe:
count    77299.000000
mean         0.714081
std          0.099089
min          0.032258
25%          0.702970
50%          0.747899
75%          0.772727
max          0.777778
dtype: float64
test  c-weight describe:
count    41778.000000
mean         0.698032
std          0.118150
min          0.000000
25%          0.680851
50%          0.739130
75%          0.769231
max          0.777778
dtype: float64

=== MoE val R² breakdown (val_fold) ===
  global only       : 0.96669
  expert only       : 0.96434
  blended (c-gated) : 0.96576
  global OOF (all)  : 0.95787
  blended OOF (all) : 0.95678


## 8. Save outputs

In [9]:
def save_pred(oof_arr, test_arr, name):
    pd.DataFrame({'Index': train['Index'].values, f'{name}_oof_pred': oof_arr}) \
      .to_parquet(os.path.join(OOF_DIR,  f'{name}.parquet'), index=False)
    pd.DataFrame({'Index': test['Index'].values, f'{name}_test_pred': test_arr}) \
      .to_parquet(os.path.join(PRED_DIR, f'{name}_test.parquet'), index=False)

save_pred(blended_oof,    blended_test,    'moe_blended')
save_pred(expert_oof_full, expert_test_full, 'moe_expert')
save_pred(global_oof,      global_test,      'moe_global')

summ = {
    'global_oof_r2': float(global_oof_r2),
    'blended_oof_r2': float(blended_oof_r2),
    'val_r2': {
        'global': float(global_val_r2),
        'expert': float(expert_val_r2),
        'blended': float(blended_val_r2),
    },
    'experts': {k: {'avg_iter': v['avg_iter'], 'best_params': v['best_params']}
                for k, v in expert_results.items()},
    'global': {'avg_iter': res_global['avg_iter'], 'best_params': res_global['best_params']},
}
with open(os.path.join(BASE, 'oof', 'moe_summary.json'), 'w') as f:
    json.dump(summ, f, indent=2, default=str)
print('Saved OOF and test predictions for moe_blended / moe_expert / moe_global.')

Saved OOF and test predictions for moe_blended / moe_expert / moe_global.
